In [3]:
import pandas as pd
import duckdb
import os
import xml.etree.ElementTree as ET

In [10]:
conn1 = duckdb.connect(database='ARTIS_SAU_v1_02-11-2025.duckdb', read_only=False)

In [11]:
# create the tables
conn1.execute("CREATE TABLE baci AS SELECT * FROM read_csv_auto('baci.csv')")
conn1.execute("CREATE TABLE code_max_resolved_taxa AS SELECT * FROM read_csv_auto('code_max_resolved.csv')")
conn1.execute("CREATE TABLE countries AS SELECT * FROM read_csv_auto('countries.csv')")
conn1.execute("CREATE TABLE products AS SELECT * FROM read_csv_auto('products.csv')")
conn1.execute("CREATE TABLE trade AS SELECT * FROM read_parquet('snet_midpoint_all_hs_all_years.parquet')")
conn1.execute("CREATE TABLE consumption AS SELECT * FROM read_parquet('2024_09_12_SAU_consumption_midpoint.parquet')")
conn1.execute("CREATE TABLE sciname AS SELECT * FROM read_csv_auto('sciname.csv')")

In [16]:
# Fetch the list of all tables
tables = conn.execute("SHOW TABLES").fetchall()

In [17]:
# Dictionary to hold schemas of all tables
schemas = {}

# Loop through each table and get its schema
for table in tables:
    table_name = table[0]  # Assuming table names are in the first column of the result
    schema_info = conn.execute(f"PRAGMA table_info('{table_name}')").fetchall()
    schemas[table_name] = schema_info

In [18]:
for table_name, schema in schemas.items():
    print(f"Schema for {table_name}:")
    for column in schema:
        print(column)

Schema for baci:
(0, 'exporter_iso3c', 'VARCHAR', False, None, False)
(1, 'importer_iso3c', 'VARCHAR', False, None, False)
(2, 'hs6', 'BIGINT', False, None, False)
(3, 'product_weight_t', 'DOUBLE', False, None, False)
(4, 'hs_version', 'VARCHAR', False, None, False)
(5, 'year', 'BIGINT', False, None, False)
Schema for code_max_resolved_taxa:
(0, 'hs_version', 'VARCHAR', False, None, False)
(1, 'hs6', 'VARCHAR', False, None, False)
(2, 'sciname', 'VARCHAR', False, None, False)
(3, 'sciname_hs_modified', 'VARCHAR', False, None, False)
Schema for consumption:
(0, 'year', 'INTEGER', False, None, False)
(1, 'hs_version', 'VARCHAR', False, None, False)
(2, 'source_country_iso3c', 'VARCHAR', False, None, False)
(3, 'exporter_iso3c', 'VARCHAR', False, None, False)
(4, 'consumer_iso3c', 'VARCHAR', False, None, False)
(5, 'sciname', 'VARCHAR', False, None, False)
(6, 'sciname_hs_modified', 'VARCHAR', False, None, False)
(7, 'habitat', 'VARCHAR', False, None, False)
(8, 'method', 'VARCHAR', False

In [19]:
# Query to fetch all rows from the baci table
query = "SELECT * FROM baci"

# Execute the query and fetch all data
baci_data = conn.execute(query).fetchdf()

# Display the data - Using pandas for better formatting and handling of large data
print(baci_data)

        exporter_iso3c importer_iso3c     hs6  product_weight_t hs_version  \
0                  AFG            SVN  160413             1.000       HS02   
1                  AGO            CHN   30379            88.200       HS02   
2                  AGO            CHN   30559             4.995       HS02   
3                  AGO            CHN   30614            68.510       HS02   
4                  AGO            CHN  160419            11.000       HS02   
...                ...            ...     ...               ...        ...   
1048570            GBR            MDA   30420            69.502       HS02   
1048571            GBR            MDA   30490           106.126       HS02   
1048572            GBR            MDA   30541            60.000       HS02   
1048573            GBR            MDA   30549             0.200       HS02   
1048574            GBR            MDA   30561             2.540       HS02   

         year  
0        2002  
1        2002  
2        2002  

In [20]:
conn1.close()